In [51]:
class Expr:
    def __init__(self, name, children=[]):
        self.name = name
        self.children = children
        self.eclass = -1

    def adjust(self, from_id, to_id):
        any_changed = False
        for i in range(len(self.children)):
            if self.children[i] == from_id:
                any_changed = True
                self.children[i] = to_id
        return any_changed

    def key(self):
        return tuple([self.name, *self.children])

    def __repr__(self):
        return f"(%s %s)" % (self.name, " ".join([c.__repr__() for c in self.children]))


class Egraph:
    def __init__(self):
        self.enodes = []
        # can this be an int
        self.eclasses = []
        self.tbl = {}

    def _fmt_node(self, key):
        """Format an enode key as a readable s-expression."""
        name, *children = key
        if not children:
            return name
        args = " ".join(f"e{self.find(c)}" for c in children)
        return f"({name} {args})"

    def __repr__(self):
        # group enode keys by canonical eclass
        from collections import defaultdict
        classes = defaultdict(list)
        for key, eid in self.tbl.items():
            classes[self.find(eid)].append(key)

        lines = []
        for eid in sorted(classes):
            nodes = " = ".join(self._fmt_node(k) for k in classes[eid])
            lines.append(f"  e{eid}: {{ {nodes} }}")
        return "Egraph(\n" + "\n".join(lines) + "\n)"

    def nodes(self, eclass_id):
        """Return list of enode keys in the given eclass."""
        canon = self.find(eclass_id)
        return [k for k, eid in self.tbl.items() if self.find(eid) == canon]

    def add(self, expr):
        # canonicalize children before inserting
        expr.children = [self.find(c) for c in expr.children]
        k = expr.key()
        if k not in self.tbl:
            self.enodes.append(expr)
            eid = len(self.eclasses)
            self.eclasses.append(eid)
            self.tbl[k] = eid
        return self.tbl[k]
    
    def find(self, a):
        while self.eclasses[a] != a:
            a = self.eclasses[a]
        return a
    
    def unify(self, a, b):
        # todo rank
        a = self.find(a)
        b = self.find(b)
        if a == b:
            return a
        self.eclasses[b] = a

        # anything that referred to b must now refer to a
        to_update = [(b, a)]
        while len(to_update) > 0:
            from_id, to_id = to_update.pop()
            for node in self.enodes:
                k = node.key()
                if k not in self.tbl:
                    continue
                changed = node.adjust(from_id, to_id)
                if changed:
                    old_eclass = self.tbl[k]

                    new_k = node.key()
                    del self.tbl[k]
                    # we need to see if it now shadows an existing expression.
                    if new_k in self.tbl:
                        f, t = old_eclass, self.tbl[new_k]
                        fc, tc = self.find(f), self.find(t)
                        if fc != tc:
                            self.eclasses[fc] = tc
                            to_update.append((fc, tc))
                    else:
                        # if not, we reuse the same eclass as before.
                        self.tbl[new_k] = old_eclass
        return a



eg = Egraph()

one = eg.add(Expr("1"))
two = eg.add(Expr("2"))

s = eg.add(Expr("+", [one, two]))

three = eg.add(Expr("3"))
ltr = eg.add(Expr("+", [one, three]))
rtl = eg.add(Expr("+", [three, one]))

eg.unify(ltr, rtl)
print("after unify(1+3, 3+1):")
print(eg)

eg.unify(s, three)
print("\nafter unify(1+2, 3):")
print(eg)
print("nodes in e2:", eg.nodes(s))

after unify(1+3, 3+1):
Egraph(
  e0: { 1 }
  e1: { 2 }
  e2: { (+ e0 e1) }
  e3: { 3 }
  e4: { (+ e0 e3) = (+ e3 e0) }
)

after unify(1+2, 3):
Egraph(
  e0: { 1 }
  e1: { 2 }
  e2: { (+ e0 e1) = 3 }
  e4: { (+ e0 e2) = (+ e2 e0) }
)
nodes in e2: [('+', 0, 1), ('3',)]


In [52]:
def test_egraph():
    # --- Basic add and dedup ---
    eg = Egraph()
    a = eg.add(Expr("a"))
    b = eg.add(Expr("b"))
    a2 = eg.add(Expr("a"))  # duplicate
    assert a == a2, "adding the same expr twice should return the same eclass"

    # --- Find before any merges ---
    assert eg.find(a) == a
    assert eg.find(b) == b

    # --- Simple unify ---
    eg.unify(a, b)
    assert eg.find(a) == eg.find(b), "unified eclasses should have same canonical id"

    # --- Congruence: merging children should merge parent nodes ---
    eg2 = Egraph()
    x = eg2.add(Expr("x"))
    y = eg2.add(Expr("y"))
    fx = eg2.add(Expr("f", [x]))
    fy = eg2.add(Expr("f", [y]))
    assert eg2.find(fx) != eg2.find(fy), "f(x) and f(y) should differ before merge"

    print(eg2)
    eg2.unify(x, y)  # x = y, so f(x) should now equal f(y)
    print(eg2)
    print(fx, fy, eg2.find(fx), eg2.find(fy))
    assert eg2.find(fx) == eg2.find(fy), (
        "congruence failure: after x=y, f(x) and f(y) should be in the same eclass"
    )

    # --- Transitive congruence ---
    eg3 = Egraph()
    a = eg3.add(Expr("a"))
    b = eg3.add(Expr("b"))
    c = eg3.add(Expr("c"))
    fa = eg3.add(Expr("f", [a]))
    fb = eg3.add(Expr("f", [b]))
    fc = eg3.add(Expr("f", [c]))
    ffa = eg3.add(Expr("f", [fa]))
    ffb = eg3.add(Expr("f", [fb]))

    eg3.unify(a, b)  # a=b => f(a)=f(b) => f(f(a))=f(f(b))
    assert eg3.find(fa) == eg3.find(fb), "congruence: f(a)=f(b) after a=b"
    assert eg3.find(ffa) == eg3.find(ffb), "deep congruence: f(f(a))=f(f(b)) after a=b"

    eg3.unify(b, c)  # now a=b=c => f(a)=f(b)=f(c)
    assert eg3.find(fa) == eg3.find(fc), "transitive congruence: f(a)=f(c) after a=b, b=c"

    # --- Multi-arg congruence ---
    eg4 = Egraph()
    p = eg4.add(Expr("p"))
    q = eg4.add(Expr("q"))
    gpq = eg4.add(Expr("g", [p, q]))
    gqp = eg4.add(Expr("g", [q, p]))
    gpp = eg4.add(Expr("g", [p, p]))

    eg4.unify(p, q)  # p=q => g(p,q)=g(q,p)=g(p,p) all congruent
    assert eg4.find(gpq) == eg4.find(gqp), "g(p,q)=g(q,p) after p=q"
    assert eg4.find(gpq) == eg4.find(gpp), "g(p,q)=g(p,p) after p=q"

    print("All tests passed!")


test_egraph()

Egraph(
  e0: { x }
  e1: { y }
  e2: { (f e0) }
  e3: { (f e1) }
)
Egraph(
  e0: { x = y }
  e2: { (f e0) }
)
2 3 2 2
All tests passed!


In [55]:
from itertools import permutations
from collections import defaultdict

anagram_indicators = [
    "cooked",
]

reverse_indicators = [
    "retracted",
    "run backwards",
]

deletion_indicators = [
    "without",
    "ignoring",
]

synonyms = [
    ("artist", "drawer"),
    ("risk", "bet"),
    ("orange", "citrus"),
    ("clever", "smart"),
    ("trains", "trams"),
    ("eastern", "e"),
    ("star", "idol"),
]

# build a lookup: word -> list of synonyms
synonym_map = {}
for a, b in synonyms:
    synonym_map.setdefault(a, []).append(b)
    synonym_map.setdefault(b, []).append(a)

# build anagram lookup: sorted letters -> list of known words
anagram_map = defaultdict(set)
known_words = set()
for a, b in synonyms:
    anagram_map[tuple(sorted(a))].add(a)
    anagram_map[tuple(sorted(b))].add(b)
    known_words.add(a)
    known_words.add(b)

def add_splits(eg, puzzle):
    splits = []
    for enode in eg.nodes(puzzle):
        if enode[0] == 'puzzle':
            for i in range(2, len(enode)):
                splits.append((enode[1:i], enode[i:]))
    for (left, right) in splits:
        left = eg.add(Expr("split", list(left)))
        right = eg.add(Expr("split", list(right)))
        joined = eg.add(Expr("chunked", [left, right]))
        eg.unify(joined, puzzle)

def add_synonyms(eg, word):
    for enode in eg.nodes(word):
        if enode[0].startswith("raw:"):
            w = enode[0][4:]
            for syn in synonym_map.get(w, []):
                syn_id = eg.add(Expr(f"derived:{syn}"))
                eg.unify(word, syn_id)

def is_indicator(eg, eclass_id, indicator_list):
    for enode in eg.nodes(eclass_id):
        if enode[0].startswith("raw:") and enode[0][4:] in indicator_list:
            return True
    return False

def is_reverse_indicator(eg, eclass_id):
    return is_indicator(eg, eclass_id, reverse_indicators)

def is_deletion_indicator(eg, eclass_id):
    return is_indicator(eg, eclass_id, deletion_indicators)

def is_anagram_indicator(eg, eclass_id):
    return is_indicator(eg, eclass_id, anagram_indicators)

def get_raw_words(eg, eclass_id):
    words = []
    for enode in eg.nodes(eclass_id):
        if enode[0].startswith("raw:"):
            words.append(enode[0][4:])
    return words

def get_all_words(eg, eclass_id):
    words = []
    for enode in eg.nodes(eclass_id):
        if enode[0].startswith("raw:") or enode[0].startswith("derived:"):
            words.append(enode[0].split(":", 1)[1])
    return words

def _consecutive_partitions(seq):
    """Generate all ways to partition seq into consecutive groups."""
    n = len(seq)
    for mask in range(1 << (n - 1)):
        partition = []
        start = 0
        for i in range(n - 1):
            if mask & (1 << i):
                partition.append(seq[start:i+1])
                start = i + 1
        partition.append(seq[start:])
        yield partition

def add_groupings(eg):
    """For each split with 3+ children, add all ways to group consecutive children."""
    for key, eid in list(eg.tbl.items()):
        if key[0] != 'split' or len(key) < 4:
            continue
        children = key[1:]
        for partition in _consecutive_partitions(children):
            if len(partition) == len(children):
                continue  # skip trivial (no grouping)
            grouped_ids = []
            for group in partition:
                if len(group) == 1:
                    grouped_ids.append(group[0])
                else:
                    # combine raw words into a single multi-word raw node
                    words = []
                    for child_id in group:
                        raw = get_raw_words(eg, child_id)
                        if raw:
                            words.append(raw[0])
                        else:
                            words = None
                            break
                    if words is None:
                        break
                    combined = " ".join(words)
                    grouped_ids.append(eg.add(Expr(f"raw:{combined}")))
            else:
                new_split = eg.add(Expr("split", grouped_ids))
                eg.unify(eg.find(eid), eg.find(new_split))

def add_reversals(eg):
    for key, eid in list(eg.tbl.items()):
        if key[0] != 'split' or len(key) != 3:
            continue
        child_a, child_b = key[1], key[2]
        for indicator, other in [(child_a, child_b), (child_b, child_a)]:
            if is_reverse_indicator(eg, indicator):
                for w in get_all_words(eg, other):
                    rev_id = eg.add(Expr(f"derived:{w[::-1]}"))
                    eg.unify(eg.find(eid), rev_id)

def delete_letters(word, letters):
    """Remove each letter in `letters` from `word` (one occurrence each)."""
    result = list(word)
    for ch in letters:
        try:
            result.remove(ch)
        except ValueError:
            return None
    return "".join(result)

def add_deletions(eg):
    for key, eid in list(eg.tbl.items()):
        if key[0] != 'split' or len(key) != 4:
            continue
        left, mid, right = key[1], key[2], key[3]
        if not is_deletion_indicator(eg, mid):
            continue
        for subject, target in [(left, right), (right, left)]:
            for sw in get_all_words(eg, subject):
                for tw in get_all_words(eg, target):
                    deleted = delete_letters(sw, tw)
                    if deleted is not None:
                        del_id = eg.add(Expr(f"derived:{deleted}"))
                        eg.unify(eg.find(eid), del_id)

def add_anagrams(eg):
    for key, eid in list(eg.tbl.items()):
        if key[0] != 'split' or len(key) < 3:
            continue
        children = key[1:]
        # try each child as the anagram indicator
        for i, child in enumerate(children):
            if not is_anagram_indicator(eg, child):
                continue
            # collect raw words from the other children
            others = [children[j] for j in range(len(children)) if j != i]
            # get all combinations of raw words and concatenate their letters
            word_lists = [get_raw_words(eg, o) for o in others]
            for combo in _product(word_lists):
                letters = "".join(combo)
                sig = tuple(sorted(letters))
                for anagram in anagram_map.get(sig, []):
                    if anagram != letters:  # don't produce the original
                        ana_id = eg.add(Expr(f"derived:{anagram}"))
                        eg.unify(eg.find(eid), ana_id)

def _product(lists):
    """Simple cartesian product of a list of lists."""
    if not lists:
        yield ()
        return
    for item in lists[0]:
        for rest in _product(lists[1:]):
            yield (item,) + rest

def reduce_singleton_splits(eg):
    for key, eid in list(eg.tbl.items()):
        if key[0] == 'split' and len(key) == 2:
            eg.unify(eg.find(eid), key[1])

def add_solutions(eg):
    for key, eid in list(eg.tbl.items()):
        if key[0] == 'chunked' and len(key) == 3:
            if eg.find(key[1]) == eg.find(key[2]):
                sol = eg.add(Expr("solution", [key[1]]))
                eg.unify(eg.find(eid), sol)

def extract_solutions(eg, root):
    solutions = []
    for enode in eg.nodes(root):
        if enode[0] == 'solution':
            child = enode[1]
            for n in eg.nodes(child):
                if n[0].startswith("derived:"):
                    word = n[0][8:]
                    if word in known_words:
                        solutions.append(word)
    return solutions

rules = [
    lambda eg, root, word_ids: add_splits(eg, root),
    lambda eg, root, word_ids: add_groupings(eg),
    lambda eg, root, word_ids: [add_synonyms(eg, wid) for wid in word_ids],
    lambda eg, root, word_ids: add_reversals(eg),
    lambda eg, root, word_ids: add_deletions(eg),
    lambda eg, root, word_ids: add_anagrams(eg),
    lambda eg, root, word_ids: reduce_singleton_splits(eg),
    lambda eg, root, word_ids: add_solutions(eg),
]

def saturate(eg, root, word_ids, max_iters=100):
    for i in range(max_iters):
        old_size = len(eg.tbl)
        for rule in rules:
            rule(eg, root, word_ids)
        if len(eg.tbl) == old_size:
            print(f"saturated after {i + 1} iterations")
            return
    print(f"stopped after {max_iters} iterations (not saturated)")

def solve_puzzle(clue):
    words = clue.split(" ")

    eg = Egraph()

    word_ids = [eg.add(Expr(f"raw:{word}")) for word in words]

    root = eg.add(Expr("puzzle", word_ids))

    saturate(eg, root, word_ids)

    solutions = extract_solutions(eg, root)
    print(f"solutions: {solutions}")
    print(eg)


solve_puzzle("cooked rustic orange")

saturated after 2 iterations
solutions: ['citrus']
Egraph(
  e1: { raw:rustic }
  e4: { raw:cooked = (split e4) }
  e5: { (split e1 e8) }
  e8: { raw:orange = derived:citrus = (split e4 e1) = (split e8) }
  e9: { (chunked e4 e5) = (puzzle e4 e1 e8) = (chunked e8 e8) = (solution e8) }
)
